# 06: Palo Alto External Validation — Native Features

Same AdapTrap **methodology** as `adaptrap_pipeline.py` (per-source-IP
behavioral profiling -> silhouette-tuned Isolation Forest contamination
-> 70th-percentile two-tier ALLOW / ESCALATE_TO_ANALYST), but every
**feature** here is built natively from what `log_4` (TRAFFIC) and
`log_5` (THREAT) actually contain -- nothing is a proxy standing in
for a CICHoneynet field Palo Alto doesn't have.

Two things these logs genuinely, directly measure that CICHoneynet
also had:
- **real timestamps** (`Receive Time`) -> `conn_rate_60s` is computed
  the same rolling-window way as `adaptrap_pipeline.compute_conn_rate`,
  on real observed times.
- **a real severity/signature system** -> THREAT log rows carry an
  actual analyst-relevant severity and named signature -- new signal
  CICHoneynet's raw captures never had at all.

Deliberately **not** included: `syn/ack/fin/rst/psh_ratio`. Palo
Alto's `Flags` field is session metadata, not TCP flags, and there's
no faithful way to recover per-packet flags from a session-summary
log -- so rather than proxy them again, they're left out entirely.

In [2]:
import sys
sys.path.insert(0, '.')
sys.path.insert(0, '/mnt/user-data/uploads')
import pandas as pd
import numpy as np

from paloalto_validation_pipeline import (
    load_and_merge, build_ip_profiles, three_way_split,
    FEATURE_COLS, TARGET_IP, is_private, SEVERITY_SCORE,
)
from adaptrap_pipeline import tune_contamination, generate_rules
from sklearn.preprocessing import StandardScaler

pd.set_option('display.width', 140)
print("Target host filtered to:", TARGET_IP)
print("Native feature columns:", FEATURE_COLS)

Target host filtered to: 202.57.49.243
Native feature columns: ['total_records', 'total_traffic_sessions', 'total_threat_alerts', 'unique_dst_ports', 'unique_applications', 'port_to_record_ratio', 'avg_bytes_per_session', 'avg_packets_per_session', 'avg_elapsed_time', 'pct_incomplete', 'pct_tcp_fin', 'pct_rst', 'pct_aged_out', 'threat_alert_rate', 'avg_severity_score', 'unique_threat_signatures', 'conn_rate_60s_max', 'conn_rate_60s_avg']


## 1. Load and merge log_4 + log_5 (unlabeled)

Keeps only records to the shared target host, drops private source
IPs (same filter `adaptrap_pipeline.is_private` applies). `Action` is
present in the raw files but not read here -- it's only used later,
for a side-by-side, never as model input.

In [5]:
merged = load_and_merge(
    r'C:\Users\Lazaro\Desktop\Enzo School Docs\test\log(4)_truncated.csv',
    r'C:\Users\Lazaro\Desktop\Enzo School Docs\test\log(5) 2.csv',
)

print("Merged records:", len(merged), "| unique source IPs:", merged['Source address'].nunique())
merged[['Receive Time','Source address','Destination address','_kind','Application']].head()

Merged records: 181 | unique source IPs: 49


,Receive Time,Source address,Destination address,_kind,Application
0,2026/09/18 10:37:11,34.78.123.91,202.57.49.243,threat,ssl
1,2026/09/18 13:59:16,172.81.61.132,202.57.49.243,threat,web-browsing
2,2026/09/18 15:43:24,20.221.75.170,202.57.49.243,threat,web-browsing
3,2026/09/18 18:09:26,198.46.87.192,202.57.49.243,threat,web-browsing
4,2026/09/18 18:09:33,198.46.87.192,202.57.49.243,threat,web-browsing


## 2. Build native per-source-IP profiles

One row per `source_ip`. Session-level stats (`avg_bytes_per_session`,
`avg_packets_per_session`, `avg_elapsed_time`, close-reason
percentages) come from TRAFFIC rows; threat stats (`threat_alert_rate`,
`avg_severity_score`, `unique_threat_signatures`) come from THREAT
rows; `conn_rate_60s_*` is computed across both logs merged, on real
timestamps.

In [6]:
profiles = build_ip_profiles(merged)
print("Profile table shape:", profiles.shape)
profiles[FEATURE_COLS].describe()

Profile table shape: (49, 19)


c:\Users\Lazaro\Desktop\Enzo School Docs\test\paloalto_validation_pipeline.py:129: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  profiles = merged.groupby("Source address").apply(_build_ip_profile).reset_index()


,total_records,total_traffic_sessions,total_threat_alerts,unique_dst_ports,unique_applications,port_to_record_ratio,avg_bytes_per_session,avg_packets_per_session,avg_elapsed_time,pct_incomplete,pct_tcp_fin,pct_rst,pct_aged_out,threat_alert_rate,avg_severity_score,unique_threat_signatures,conn_rate_60s_max,conn_rate_60s_avg
count,49.000000,49.000000,49.000000,49.0,49.0,49.000000,4.900000e+01,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000
mean,3.693878,2.775510,0.918367,1.0,1.0,0.839213,4.987548e+04,14.441102,3.530682,0.040816,0.183673,0.020408,0.244898,0.551020,1.061224,0.551020,3.367347,2.949511
std,13.741974,13.832729,1.643633,0.0,0.0,0.314095,2.800347e+05,46.749617,15.742551,0.199915,0.391230,0.142857,0.434483,0.502545,1.297564,0.502545,13.722995,11.876628
min,1.000000,0.000000,0.000000,1.0,1.0,0.010309,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000
25%,1.000000,0.000000,0.000000,1.0,1.0,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000
50%,1.000000,0.000000,1.000000,1.0,1.0,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
75%,1.000000,1.000000,1.000000,1.0,1.0,1.000000,8.700556e+03,8.833333,0.333333,0.000000,0.000000,0.000000,0.000000,1.000000,2.000000,1.000000,1.000000,1.000000
max,97.000000,97.000000,8.000000,1.0,1.0,1.000000,1.960857e+06,304.000000,90.000000,1.000000,1.000000,1.000000,1.000000,1.000000,4.000000,1.000000,97.000000,84.164948


## 3. Train / validation / holdout split

Plain random 70/15/15 split (same fallback `adaptrap_pipeline` uses
when a dataset is too small/uniform to stratify meaningfully).

In [7]:
train_idx, val_idx, test_idx = three_way_split(profiles)
print(f"train={len(train_idx)}  val={len(val_idx)}  holdout={len(test_idx)}")

scaler = StandardScaler().fit(profiles.iloc[train_idx][FEATURE_COLS])
X_train = scaler.transform(profiles.iloc[train_idx][FEATURE_COLS])
X_val = scaler.transform(profiles.iloc[val_idx][FEATURE_COLS])

train=34  val=7  holdout=8


## 4. Tune contamination (silhouette-selected, validation set only)

Same selection logic as `adaptrap_pipeline.tune_contamination` --
purely unsupervised. Compare this sweep to the proxied-feature version
from the previous notebook: there every candidate flagged 0 points and
silhouette was `NaN` straight down the column. With native features
there should be real variance and a genuine peak.

In [8]:
contamination, model, sweep = tune_contamination(X_train, X_val)
print(sweep)
print("\nSelected contamination:", contamination)

    contamination  flagging_rate  silhouette
0          0.0100       0.285714    0.539315
1          0.0125       0.285714    0.539315
2          0.0150       0.285714    0.539315
3          0.0175       0.285714    0.539315
4          0.0200       0.285714    0.539315
5          0.0500       0.285714    0.539315
6          0.0750       0.285714    0.539315
7          0.1000       0.285714    0.539315
8          0.1250       0.285714    0.539315
9          0.1500       0.428571    0.274106
10         0.2000       0.428571    0.274106

Selected contamination: 0.01


## 5. Score the FULL population and generate ALLOW / ESCALATE_TO_ANALYST

Every IP (train, val, holdout) gets scored so the output can sit next
to `Action` for every record. `split` on the output flags which IPs
were genuinely held out vs. seen during fit/tune -- only `holdout`
rows are a leakage-free read.

In [9]:
X_all = scaler.transform(profiles[FEATURE_COLS])

port_lookup = merged.groupby('Source address')['Destination Port'] \
    .agg(lambda x: x.mode().iloc[0] if len(x.mode()) else None).to_dict()

rules_df = generate_rules(model, X_all, profiles.reset_index(drop=True), port_lookup,
                           label="Palo Alto NATIVE features (FULL population)")

split_label = {}
for i in train_idx: split_label[profiles.iloc[i]['source_ip']] = 'train'
for i in val_idx: split_label[profiles.iloc[i]['source_ip']] = 'val'
for i in test_idx: split_label[profiles.iloc[i]['source_ip']] = 'holdout'
rules_df['split'] = rules_df['source_ip'].map(split_label)
rules_df.head(10)


--- Palo Alto NATIVE features (FULL population) ---
         source_ip  port  anomaly_score               action
0   110.54.132.233   443       0.016138  ESCALATE_TO_ANALYST
1    198.46.87.192   443       0.004995  ESCALATE_TO_ANALYST
2   129.227.97.123   443       0.003490  ESCALATE_TO_ANALYST
3    47.128.35.211   443      -0.007086  ESCALATE_TO_ANALYST
4   17.166.235.171   443      -0.039698  ESCALATE_TO_ANALYST
5   46.151.178.133   443      -0.061229  ESCALATE_TO_ANALYST
6     13.52.79.116   443      -0.080363  ESCALATE_TO_ANALYST
7    213.230.86.78   443      -0.083930  ESCALATE_TO_ANALYST
8   172.94.125.133   443      -0.097587  ESCALATE_TO_ANALYST
9  206.123.152.180   443      -0.101215  ESCALATE_TO_ANALYST

ESCALATE_TO_ANALYST: 15, ALLOW: 34
(No records are auto-blocked. BLOCK is an analyst decision made after reviewing an ESCALATE_TO_ANALYST record, not an automated label.)


,source_ip,port,anomaly_score,action,split
0,110.54.132.233,443,0.016138,ESCALATE_TO_ANALYST,val
1,198.46.87.192,443,0.004995,ESCALATE_TO_ANALYST,val
2,129.227.97.123,443,0.003490,ESCALATE_TO_ANALYST,train
3,47.128.35.211,443,-0.007086,ESCALATE_TO_ANALYST,train
4,17.166.235.171,443,-0.039698,ESCALATE_TO_ANALYST,train
5,46.151.178.133,443,-0.061229,ESCALATE_TO_ANALYST,train
6,13.52.79.116,443,-0.080363,ESCALATE_TO_ANALYST,train
7,213.230.86.78,443,-0.083930,ESCALATE_TO_ANALYST,holdout
8,172.94.125.133,443,-0.097587,ESCALATE_TO_ANALYST,val
9,206.123.152.180,443,-0.101215,ESCALATE_TO_ANALYST,train


## 6. Export per-IP results

The model's actual unit of decision -- one row per source IP, 49
decisions total.

In [10]:
rules_df.to_csv('paloalto_native_rules.csv', index=False)
print("Exported:", len(rules_df), "IPs -> paloalto_native_rules.csv")
print(rules_df['action'].value_counts())

Exported: 49 IPs -> paloalto_native_rules.csv
action
ALLOW                  34
ESCALATE_TO_ANALYST    15
Name: count, dtype: int64


## 7. Broadcast to per-record view, side by side with `Action`

Reshape only -- the model still only made 49 decisions. Each original
log line inherits its source IP's action/score so it can sit next to
that record's real `Action`. No derived ground-truth column, no
agreement score computed here -- this is for eyeballing, not grading.

In [11]:
ip_to_action = dict(zip(rules_df['source_ip'], rules_df['action']))
ip_to_score = dict(zip(rules_df['source_ip'], rules_df['anomaly_score']))
ip_to_split = dict(zip(rules_df['source_ip'], rules_df['split']))

per_record = merged[['Receive Time','Source address','Destination address',
                      'Source Port','Destination Port','_kind','Action']].copy()
per_record = per_record.rename(columns={
    'Source address': 'source_ip', 'Destination address': 'target_ip',
    '_kind': 'log_type', 'Action': 'firewall_action',
})
per_record['model_action'] = per_record['source_ip'].map(ip_to_action)
per_record['model_anomaly_score'] = per_record['source_ip'].map(ip_to_score)
per_record['model_split'] = per_record['source_ip'].map(ip_to_split)
per_record = per_record.sort_values('Receive Time').reset_index(drop=True)

per_record.to_csv('paloalto_native_rules_per_record.csv', index=False)
print("Exported:", len(per_record), "records -> paloalto_native_rules_per_record.csv")
per_record.head(10)

Exported: 181 records -> paloalto_native_rules_per_record.csv


,Receive Time,source_ip,target_ip,Source Port,Destination Port,log_type,firewall_action,model_action,model_anomaly_score,model_split
0,2026/09/18 10:37:11,34.78.123.91,202.57.49.243,16350,80,threat,alert,ALLOW,-0.326700,train
1,2026/09/18 13:59:16,172.81.61.132,202.57.49.243,56032,80,threat,alert,ALLOW,-0.258673,train
2,2026/09/18 15:43:24,20.221.75.170,202.57.49.243,60024,80,threat,reset-both,ALLOW,-0.323094,holdout
3,2026/09/18 18:09:26,198.46.87.192,202.57.49.243,45832,443,threat,reset-both,ESCALATE_TO_ANALYST,0.004995,val
4,2026/09/18 18:09:33,198.46.87.192,202.57.49.243,36018,443,threat,reset-both,ESCALATE_TO_ANALYST,0.004995,val
5,2026/09/18 18:09:40,198.46.87.192,202.57.49.243,48234,443,threat,reset-both,ESCALATE_TO_ANALYST,0.004995,val
6,2026/09/18 18:09:49,198.46.87.192,202.57.49.243,48268,443,threat,reset-both,ESCALATE_TO_ANALYST,0.004995,val
7,2026/09/19 02:09:17,198.46.87.192,202.57.49.243,45190,443,threat,reset-both,ESCALATE_TO_ANALYST,0.004995,val
8,2026/09/19 02:09:23,198.46.87.192,202.57.49.243,56996,443,threat,reset-both,ESCALATE_TO_ANALYST,0.004995,val
9,2026/09/19 02:09:31,198.46.87.192,202.57.49.243,50630,443,threat,reset-both,ESCALATE_TO_ANALYST,0.004995,val


## 8. Holdout match rate against the firewall's own actions

Informal comparison only -- `Action` collapsed to ALLOW vs. not-ALLOW,
majority-voted per IP where an IP has multiple records, then compared
to the model's per-IP action. This is a sanity read, not a formal
evaluation (see the caveats below).

In [12]:
merged['fw_side'] = merged['Action'].map(lambda a: 'ALLOW' if a == 'allow' else 'ESCALATE_TO_ANALYST')
ip_majority = merged.groupby('Source address')['fw_side'].agg(lambda s: s.value_counts().idxmax())

rules_df['fw_majority'] = rules_df['source_ip'].map(ip_majority)
rules_df['match'] = rules_df['action'] == rules_df['fw_majority']

def pct(df, label):
    print(f"{label}: {df['match'].mean()*100:.1f}%  (n={len(df)}, matches={df['match'].sum()})")

pct(rules_df, 'Per-IP, full population (49 IPs)')
pct(rules_df[rules_df['split'] == 'holdout'], 'Per-IP, holdout only')

Per-IP, full population (49 IPs): 30.6%  (n=49, matches=15)
Per-IP, holdout only: 25.0%  (n=8, matches=2)


## 9. Summary

Native features fixed the **tuning pathology** the proxied-feature
version had -- the contamination sweep now shows real variance and a
genuine silhouette peak instead of `NaN` across every candidate.

They did **not** fix the **sample-size problem** -- only 8 IPs are
genuinely held out, so the match-rate number above (whatever it reads)
should be treated as too small to generalize from; one flipped IP
moves it by 12.5 percentage points.

Two separate, non-overlapping findings, and worth keeping them
separate when writing this up:
1. Proxied/forced CICHoneynet-shaped features broke the model's own
   contamination-tuning step outright (not just accuracy).
2. Even with native, faithful features, 8 holdout IPs is too small a
   sample to draw a confident conclusion either way.